# Legacy competition notebook (archived, not maintained)

This is the original TEKNOFEST 2026 competition prototype, kept for history. Installation commands, execution outputs and the temporary demo link have been removed; the code is otherwise unchanged. **It is not the implementation of this repository** — see `src/varpath/` and the README.

## Methodological audit

The evaluation in this notebook is optimistic and its numbers are not used as the repository's estimate:

1. **Grouping.** `hash(X_raw) % 200` groups rows by numeric profile. The panel files share 578 variants (same `Variant_ID`, label and residue change, slightly perturbed annotations), which this hash does not group, so the same variant could sit on both sides of a split.
2. **Hyperparameter search.** The XGBoost and LightGBM Optuna objectives score the outer validation fold.
3. **Early stopping.** LightGBM early stopping and FT-Transformer epoch selection (`roc_auc_score(y_val, ...)`) use the outer validation fold.
4. **Tuned parameters were not applied.** `XGBClassifier(**study.best_params)` receives the Optuna keys `n_est`, `lr`, `md`, ..., which XGBoost and LightGBM silently ignore; the final fold models used library defaults.
5. **Stacking.** The logistic-regression meta learner is fitted on all OOF meta-features and evaluated on the same rows.
6. **Thresholds.** The decision threshold (overall, per fold and per panel) is optimised and reported on the same predictions.
7. **Global preprocessing.** One-hot encoders, missing-indicator selection (mutual information with all labels), conflict removal, class weights and TDA median filling use the full dataset.
8. **Panel fine-tuning.** The base fold is chosen by validation AUC and thresholds are tuned on the panel validation subset.
9. **Calibration.** Isotonic regression is fitted on a random 30% subset and evaluated on all rows including that subset.
10. **Demo.** Only two residues are user inputs; the remaining ~770 features are filled with medians/zeros, and a fixed 0.514 threshold appears in the PDF report next to a different model threshold.

Legacy competition run — methodologically optimistic, not used as the repository's primary estimate: stacking OOF ROC-AUC 0.8633, PR-AUC 0.9345, F2 0.9136, MCC 0.5662 (A100 GPU, Colab).


In [ ]:
"""
VarPath — TEKNOFEST 2026 Sağlıkta Yapay Zeka
Missense Varyant Patojenisite Sınıflandırması
v2: TDA genişletme, yapısal özellikler, varyant ağı,
    fenotip bağlantısı, bootstrap CI, test inference
"""

import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, fbeta_score,
    precision_recall_curve, confusion_matrix, matthews_corrcoef,
    balanced_accuracy_score, roc_curve, brier_score_loss, log_loss
)
import xgboost as xgb
import lightgbm as lgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

try:
    from ripser import ripser
    TDA_OK = True
    print("[TDA] ripser yüklü ✓")
except ImportError:
    TDA_OK = False
    print("[TDA] ripser yok — pip install ripser")

try:
    import networkx as nx
    NX_OK = True
except ImportError:
    NX_OK = False
    print("[NETWORK] networkx yok — pip install networkx")

from fpdf import FPDF

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[DEVICE] {DEVICE}")

PANELS    = ['master', 'kanser', 'pah', 'cftr']
PANEL2IDX = {p: i for i, p in enumerate(PANELS)}
N_FOLDS   = 5
MIN_SPEC  = 0.60
OPTUNA_N  = 25
FT_EPOCHS = 50
FT_LR     = 3e-4
FT_BATCH  = 64
D_TOKEN   = 64
N_HEADS   = 4
N_LAYERS  = 2
MTL_ALPHA = 0.7
N_BOOT    = 1000  # Bootstrap iterasyon sayısı

# ════════════════════════════════════════════════════════
# FENOTIP HARITALAMA
# ════════════════════════════════════════════════════════
PANEL_PHENOTYPE = {
    'master': 'Genel Genetik Panel — Cok Hastalikli Tarama',
    'kanser': 'Onkoloji / Kanser Yatkınlığı (BRCA, TP53 vb.)',
    'pah'   : 'Fenilketonüri (PKU) / PAH Enzim Eksikligi',
    'cftr'  : 'Kistik Fibrozis (CF) / CFTR Kanal Bozuklugu',
}

PANEL_ACTION = {
    'master': 'Genetik konsültasyon ve panel genisletme onerilir.',
    'kanser': 'Onkogenetik danısmanlik ve aile taraması onerilir.',
    'pah'   : 'Diyetisyen yonlendirmesi ve fenilalanin izlemi onerilir.',
    'cftr'  : 'Solunum fonksiyon testi ve mukolitik tedavi degerlendirmesi onerilir.',
}

# ════════════════════════════════════════════════════════
# 1. AMİNO ASİT ÖZELLİKLERİ
# ════════════════════════════════════════════════════════

GRANTHAM = {
    ('A','R'):112,('A','N'):111,('A','D'):126,('A','C'):195,('A','Q'):91,
    ('A','E'):107,('A','G'):60, ('A','H'):86, ('A','I'):94, ('A','L'):96,
    ('A','K'):106,('A','M'):84, ('A','F'):113,('A','P'):27, ('A','S'):99,
    ('A','T'):58, ('A','W'):148,('A','Y'):112,('A','V'):64,
    ('R','N'):86, ('R','D'):96, ('R','C'):180,('R','Q'):43, ('R','E'):54,
    ('R','G'):125,('R','H'):29, ('R','I'):97, ('R','L'):102,('R','K'):26,
    ('R','M'):91, ('R','F'):97, ('R','P'):103,('R','S'):110,('R','T'):71,
    ('R','W'):101,('R','Y'):77, ('R','V'):96,
    ('N','D'):23, ('N','C'):139,('N','Q'):46, ('N','E'):42, ('N','G'):80,
    ('N','H'):68, ('N','I'):149,('N','L'):153,('N','K'):94, ('N','M'):142,
    ('N','F'):158,('N','P'):91, ('N','S'):46, ('N','T'):65, ('N','W'):174,
    ('N','Y'):143,('N','V'):133,
    ('D','C'):154,('D','Q'):61, ('D','E'):45, ('D','G'):94, ('D','H'):81,
    ('D','I'):168,('D','L'):172,('D','K'):101,('D','M'):160,('D','F'):177,
    ('D','P'):108,('D','S'):65, ('D','T'):85, ('D','W'):181,('D','Y'):160,
    ('D','V'):152,
    ('C','Q'):154,('C','E'):170,('C','G'):159,('C','H'):174,('C','I'):198,
    ('C','L'):198,('C','K'):202,('C','M'):196,('C','F'):205,('C','P'):169,
    ('C','S'):112,('C','T'):149,('C','W'):215,('C','Y'):194,('C','V'):192,
    ('Q','E'):29, ('Q','G'):87, ('Q','H'):24, ('Q','I'):109,('Q','L'):113,
    ('Q','K'):53, ('Q','M'):101,('Q','F'):116,('Q','P'):76, ('Q','S'):68,
    ('Q','T'):42, ('Q','W'):130,('Q','Y'):99, ('Q','V'):96,
    ('E','G'):98, ('E','H'):40, ('E','I'):134,('E','L'):138,('E','K'):56,
    ('E','M'):126,('E','F'):140,('E','P'):93, ('E','S'):80, ('E','T'):65,
    ('E','W'):152,('E','Y'):122,('E','V'):121,
    ('G','H'):98, ('G','I'):135,('G','L'):138,('G','K'):127,('G','M'):127,
    ('G','F'):153,('G','P'):42, ('G','S'):56, ('G','T'):59, ('G','W'):184,
    ('G','Y'):147,('G','V'):109,
    ('H','I'):94, ('H','L'):99, ('H','K'):32, ('H','M'):87, ('H','F'):100,
    ('H','P'):77, ('H','S'):89, ('H','T'):47, ('H','W'):115,('H','Y'):83,
    ('H','V'):84,
    ('I','L'):5,  ('I','K'):102,('I','M'):10, ('I','F'):21, ('I','P'):95,
    ('I','S'):142,('I','T'):89, ('I','W'):61, ('I','Y'):33, ('I','V'):29,
    ('L','K'):107,('L','M'):15, ('L','F'):22, ('L','P'):98, ('L','S'):145,
    ('L','T'):92, ('L','W'):61, ('L','Y'):36, ('L','V'):32,
    ('K','M'):95, ('K','F'):102,('K','P'):103,('K','S'):121,('K','T'):78,
    ('K','W'):110,('K','Y'):85, ('K','V'):97,
    ('M','F'):28, ('M','P'):87, ('M','S'):135,('M','T'):81, ('M','W'):67,
    ('M','Y'):36, ('M','V'):21,
    ('F','P'):114,('F','S'):155,('F','T'):103,('F','W'):40, ('F','Y'):22,
    ('F','V'):50,
    ('P','S'):74, ('P','T'):38, ('P','W'):147,('P','Y'):110,('P','V'):68,
    ('S','T'):58, ('S','W'):177,('S','Y'):144,('S','V'):124,
    ('T','W'):128,('T','Y'):92, ('T','V'):69,
    ('W','Y'):37, ('W','V'):88,
    ('Y','V'):55,
}
_sym = {(b,a): v for (a,b),v in GRANTHAM.items()}
GRANTHAM.update(_sym)

AA_PROPS = {
    'A':[ 1.80,6.00, 89.09, 67,0, 0,0,0.36],
    'R':[-4.50,10.76,174.20,148,1, 1,0,0.53],
    'N':[-3.50,5.41,132.12, 96,1, 0,0,0.46],
    'D':[-3.50,2.77,133.10, 91,1,-1,0,0.51],
    'C':[ 2.50,5.07,121.16, 86,0, 0,0,0.35],
    'E':[-3.50,3.22,147.13,109,1,-1,0,0.50],
    'Q':[-3.50,5.65,146.15,114,1, 0,0,0.49],
    'G':[-0.40,5.97, 75.03, 48,0, 0,0,0.54],
    'H':[-3.20,7.59,155.16,118,1, 0,1,0.32],
    'I':[ 4.50,6.02,131.17,124,0, 0,0,0.46],
    'L':[ 3.80,5.98,131.17,124,0, 0,0,0.45],
    'K':[-3.90,9.74,146.19,135,1, 1,0,0.47],
    'M':[ 1.90,5.74,149.21,124,0, 0,0,0.44],
    'F':[ 2.80,5.48,165.19,135,0, 0,1,0.31],
    'P':[-1.60,6.30,115.13, 90,0, 0,0,0.51],
    'S':[-0.80,5.68,105.09, 73,1, 0,0,0.51],
    'T':[-0.70,5.60,119.12, 93,1, 0,0,0.44],
    'W':[-0.90,5.89,204.23,163,0, 0,1,0.31],
    'Y':[-1.30,5.66,181.19,141,1, 0,1,0.42],
    'V':[ 4.20,5.96,117.15,105,0, 0,0,0.39],
}
PROP_NAMES = ['hydro','pI','MW','vol','polar','charge','arom','flex']

_B62_ROWS = 'A R N D C Q E G H I L K M F P S T W Y V'.split()
_B62_DATA = [
    [ 4,-1,-2,-2, 0,-1,-1, 0,-2,-1,-1,-1,-1,-2,-1, 1, 0,-3,-2, 0],
    [-1, 5, 0,-2,-3, 1, 0,-2, 0,-3,-2, 2,-1,-3,-2,-1,-1,-3,-2,-3],
    [-2, 0, 6, 1,-3, 0, 0, 0, 1,-3,-3, 0,-2,-3,-2, 1, 0,-4,-2,-3],
    [-2,-2, 1, 6,-3, 0, 2,-1,-1,-3,-4,-1,-3,-3,-1, 0,-1,-4,-3,-3],
    [ 0,-3,-3,-3, 9,-3,-4,-3,-3,-1,-1,-3,-1,-2,-3,-1,-1,-2,-2,-1],
    [-1, 1, 0, 0,-3, 5, 2,-2, 0,-3,-2, 1, 0,-3,-1, 0,-1,-2,-1,-2],
    [-1, 0, 0, 2,-4, 2, 5,-2, 0,-3,-3, 1,-2,-3,-1, 0,-1,-3,-2,-2],
    [ 0,-2, 0,-1,-3,-2,-2, 6,-2,-4,-4,-2,-3,-3,-2, 0,-2,-2,-3,-3],
    [-2, 0, 1,-1,-3, 0, 0,-2, 8,-3,-3,-1,-2,-1,-2,-1,-2,-2, 2,-3],
    [-1,-3,-3,-3,-1,-3,-3,-4,-3, 4, 2,-3, 1, 0,-3,-2,-1,-3,-1, 3],
    [-1,-2,-3,-4,-1,-2,-3,-4,-3, 2, 4,-2, 2, 0,-3,-2,-1,-2,-1, 1],
    [-1, 2, 0,-1,-3, 1, 1,-2,-1,-3,-2, 5,-1,-3,-1, 0,-1,-3,-2,-2],
    [-1,-1,-2,-3,-1, 0,-2,-3,-2, 1, 2,-1, 5, 0,-2,-1,-1,-1,-1, 1],
    [-2,-3,-3,-3,-2,-3,-3,-3,-1, 0, 0,-3, 0, 6,-4,-2,-2, 1, 3,-1],
    [-1,-2,-2,-1,-3,-1,-1,-2,-2,-3,-3,-1,-2,-4, 7,-1,-1,-4,-3,-2],
    [ 1,-1, 1, 0,-1, 0, 0, 0,-1,-2,-2, 0,-1,-2,-1, 4, 1,-3,-2,-2],
    [ 0,-1, 0,-1,-1,-1,-1,-2,-2,-1,-1,-1,-1,-2,-1, 1, 5,-2,-2, 0],
    [-3,-3,-4,-4,-2,-2,-3,-2,-2,-3,-2,-3,-1, 1,-4,-3,-2,11, 2,-3],
    [-2,-2,-2,-3,-2,-1,-2,-3, 2,-1,-1,-2,-1, 3,-3,-2,-2, 2, 7,-1],
    [ 0,-3,-3,-3,-1,-2,-2,-3,-3, 3, 1,-2, 1,-1,-2,-2, 0,-3,-1, 4],
]
BLOSUM62 = {(r,c): _B62_DATA[i][j]
            for i,r in enumerate(_B62_ROWS)
            for j,c in enumerate(_B62_ROWS)}

# ════════════════════════════════════════════════════════
# YENİ: YAPISAL ÖZELLİKLER (Protein Yapı Tahmini)
# Chou-Fasman ikincil yapı eğilim parametreleri +
# normalize solvent accessibility + rigidity index
# ════════════════════════════════════════════════════════

# Pa: helix, Pb: sheet, Pt: turn eğilimleri (Chou-Fasman 1978)
AA_STRUCT = {
    #    Pa     Pb     Pt    RSA    rigid
    'A':[1.42, 0.83, 0.66, 0.50,  0.72],
    'R':[0.98, 0.93, 0.95, 0.83,  0.45],
    'N':[0.67, 0.89, 1.56, 0.82,  0.52],
    'D':[1.01, 0.54, 1.46, 0.81,  0.55],
    'C':[0.70, 1.19, 1.19, 0.40,  0.88],
    'E':[1.51, 0.37, 0.74, 0.84,  0.50],
    'Q':[1.11, 1.10, 0.98, 0.81,  0.48],
    'G':[0.57, 0.75, 1.56, 0.51,  1.00],
    'H':[1.00, 0.87, 0.95, 0.72,  0.62],
    'I':[1.08, 1.60, 0.47, 0.37,  0.78],
    'L':[1.21, 1.30, 0.59, 0.37,  0.75],
    'K':[1.16, 0.74, 1.01, 0.93,  0.38],
    'M':[1.45, 1.05, 0.60, 0.44,  0.68],
    'F':[1.13, 1.38, 0.60, 0.35,  0.80],
    'P':[0.57, 0.55, 1.52, 0.53,  1.00],
    'S':[0.77, 0.75, 1.43, 0.64,  0.62],
    'T':[0.83, 1.19, 0.96, 0.61,  0.65],
    'W':[1.08, 1.37, 0.96, 0.28,  0.75],
    'Y':[0.69, 1.47, 1.14, 0.47,  0.70],
    'V':[1.06, 1.70, 0.50, 0.36,  0.80],
}
STRUCT_NAMES = ['Pa','Pb','Pt','RSA','rigid']

def structural_features(aa1, aa2):
    """Protein yapısal özellikler: Chou-Fasman + solvent accessibility + rigidity"""
    z = np.zeros(5)
    s1 = np.array(AA_STRUCT.get(aa1, z))
    s2 = np.array(AA_STRUCT.get(aa2, z))
    diff = s2 - s1
    # Yapısal bozunum skoru: büyük fark = daha büyük yapısal etki
    disruption = np.linalg.norm(diff)
    # İkincil yapı değişimi: helix→sheet veya tam tersi
    helix_change = abs(s2[0] - s1[0])
    sheet_change = abs(s2[1] - s1[1])
    return np.concatenate([s1, s2, diff,
                           [disruption, helix_change, sheet_change]])  # 17

STRUCT_FEAT_NAMES = (
    [f'str1_{n}' for n in STRUCT_NAMES] +
    [f'str2_{n}' for n in STRUCT_NAMES] +
    [f'dstr_{n}' for n in STRUCT_NAMES] +
    ['struct_disruption', 'helix_change', 'sheet_change']
)

def aa_features(aa1, aa2):
    z = np.zeros(8)
    p1 = np.array(AA_PROPS.get(aa1, z))
    p2 = np.array(AA_PROPS.get(aa2, z))
    return np.concatenate([p1, p2, p2-p1,
                           [float(BLOSUM62.get((aa1,aa2),0)),
                            float(GRANTHAM.get((aa1,aa2),0))]])

AA_FEAT_NAMES = ([f'aa1_{n}' for n in PROP_NAMES] +
                 [f'aa2_{n}' for n in PROP_NAMES] +
                 [f'd_{n}'   for n in PROP_NAMES] +
                 ['blosum62','grantham'])

# ════════════════════════════════════════════════════════
# 2. KLİNİK RAPOR — fenotip bağlantısı eklendi
# ════════════════════════════════════════════════════════

class PDF(FPDF):
    def __init__(self):
        super().__init__()
        self.add_font('DejaVu', '',  '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', uni=True)
        self.add_font('DejaVu', 'B', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', uni=True)
        self.add_font('DejaVu', 'I', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Oblique.ttf', uni=True)

def klinik_rapor_uret(variant_id, panel, proba, shap_values,
                       feature_names, aa1, aa2, grantham,
                       ci_lower=None, ci_upper=None):
    karar = "PATOJENIK" if proba >= 0.514 else "BENIGN"
    guven = "YUKSEK" if abs(proba - 0.5) > 0.25 else "ORTA"

    top3_idx = np.argsort(np.abs(shap_values))[-3:][::-1]
    top3 = [(feature_names[i], shap_values[i]) for i in top3_idx]

    phenotype = PANEL_PHENOTYPE.get(panel, panel.upper())
    action    = PANEL_ACTION.get(panel, 'Klinisyen degerlendirmesi onerilir.')

    # Yapısal analiz
    s1 = np.array(AA_STRUCT.get(aa1, np.zeros(5)))
    s2 = np.array(AA_STRUCT.get(aa2, np.zeros(5)))
    struct_disruption = float(np.linalg.norm(s2 - s1))
    struct_level = "yuksek" if struct_disruption > 0.8 else ("orta" if struct_disruption > 0.4 else "dusuk")

    pdf = PDF()
    pdf.add_page()

    # Başlık
    pdf.set_font("DejaVu", "B", 16)
    pdf.cell(0, 10, "VarPath Genetik Varyant Raporu", ln=True, align="C")
    pdf.set_font("DejaVu", size=9)
    pdf.cell(0, 6, "VarPath | TEKNOFEST 2026 Saglıkta Yapay Zeka", ln=True, align="C")
    pdf.ln(2)

    # Temel bilgiler
    pdf.set_font("DejaVu", "B", 12)
    pdf.cell(0, 8, "Varyant Bilgileri", ln=True)
    pdf.set_font("DejaVu", size=11)
    pdf.cell(0, 7, f"Varyant ID   : {variant_id}", ln=True)
    pdf.cell(0, 7, f"Panel        : {panel.upper()}", ln=True)
    pdf.cell(0, 7, f"Fenotip      : {phenotype}", ln=True)
    pdf.cell(0, 7, f"AA Degisimi  : {aa1} -> {aa2}", ln=True)

    if grantham > 100:
        gran_yorum = "yuksek - radikal degisim"
    elif grantham > 50:
        gran_yorum = "orta - fizikokimyasal fark var"
    else:
        gran_yorum = "dusuk - benzer amino asitler"
    pdf.cell(0, 7, f"Grantham     : {grantham} ({gran_yorum})", ln=True)
    pdf.cell(0, 7, f"Yapisal Etki : {struct_level} (disruption={struct_disruption:.2f})", ln=True)
    pdf.ln(2)

    # Karar kutusu
    if karar == "PATOJENIK":
        pdf.set_fill_color(220, 50, 50)
    else:
        pdf.set_fill_color(50, 180, 50)
    pdf.set_font("DejaVu", "B", 14)
    pdf.cell(0, 12, f"KARAR: {karar} (Olasilik: {proba:.1%})",
             ln=True, fill=True, align="C")
    pdf.ln(1)

    pdf.set_font("DejaVu", size=11)
    pdf.cell(0, 7, f"Guven Seviyesi: {guven}", ln=True)
    pdf.ln(2)

    # SHAP faktörler
    pdf.set_font("DejaVu", "B", 12)
    pdf.cell(0, 7, "Karari Etkileyen Baslica Faktorler (SHAP):", ln=True)
    pdf.set_font("DejaVu", size=11)
    for fname, fval in top3:
        yon = "patojenik yonunde" if fval > 0 else "benign yonunde"
        pdf.cell(0, 6, f"  • {fname}: {yon} (etki: {fval:+.3f})", ln=True)
    pdf.ln(2)

    # Fenotip ve klinik öneri
    pdf.set_font("DejaVu", "B", 12)
    pdf.cell(0, 7, "Klinik Yorum ve Oneriler:", ln=True)
    pdf.set_font("DejaVu", "I", 10)
    if karar == "PATOJENIK":
        not_text = f"Bu varyant yuksek patojenisite skoru almistir. {action}"
    else:
        not_text = f"Bu varyant benign olarak siniflandirilmistir. Klinisyen degerlendirmesi onerilir."
    pdf.multi_cell(0, 6, not_text)

    pdf.output(f"rapor_{variant_id}.pdf")
    print(f"  Rapor olusturuldu: rapor_{variant_id}.pdf")

# ════════════════════════════════════════════════════════
# 3. TDA — GENİŞLETİLMİŞ (itiraz kapsamı)
# Betti sayıları, persistence entropy, dağılım özetleri
# ════════════════════════════════════════════════════════

def _pers_stats(dgm):
    """H0 veya H1 persistence diyagramından istatistikler"""
    if len(dgm) == 0: return [0., 0., 0., 0., 0.]
    p = dgm[:,1] - dgm[:,0]; p = p[np.isfinite(p) & (p > 0)]
    if len(p) == 0: return [0., 0., 0., 0., 0.]
    s = p.sum() + 1e-12
    entropy = float(-np.sum((p/s) * np.log(p/s + 1e-12)))
    return [
        float(np.mean(p)),           # ortalama persistence
        float(np.max(p)),            # maksimum persistence
        entropy,                     # persistence entropy
        float(p.sum()),              # toplam persistence
        float(np.std(p)) if len(p)>1 else 0.  # std persistence
    ]

def tda_features(signal, n_pts=50):
    """
    Genisletilmis TDA: Betti sayıları (b0, b1),
    H0/H1 icin mean, max, entropy, total, std
    + Euler karakteristigi
    """
    if not TDA_OK: return np.zeros(13, dtype=np.float32)
    sig = signal[np.isfinite(signal)]
    if len(sig) < 4: return np.zeros(13, dtype=np.float32)
    if len(sig) > n_pts:
        sig = sig[np.linspace(0, len(sig)-1, n_pts, dtype=int)]
    pts = np.column_stack([sig[:-1], sig[1:]])
    try:
        res  = ripser(pts, maxdim=1)
        dgm0 = res['dgms'][0]; dgm1 = res['dgms'][1]
        # Betti sayıları: sonsuza kadar yaşayan özellikler
        b0 = float(np.sum(np.isfinite(dgm0[:,1]))) if len(dgm0) else 0.
        b1 = float(np.sum(np.isfinite(dgm1[:,1]))) if len(dgm1) else 0.
        euler = b0 - b1  # Euler karakteristigi
        h0_stats = _pers_stats(dgm0)
        h1_stats = _pers_stats(dgm1)
        return np.array([b0, b1, euler] + h0_stats + h1_stats, dtype=np.float32)
    except:
        return np.zeros(13, dtype=np.float32)

TDA_NAMES = [
    'tda_b0', 'tda_b1', 'tda_euler',
    'tda_h0_mean', 'tda_h0_max', 'tda_h0_entropy', 'tda_h0_total', 'tda_h0_std',
    'tda_h1_mean', 'tda_h1_max', 'tda_h1_entropy', 'tda_h1_total', 'tda_h1_std',
]

# ════════════════════════════════════════════════════════
# 4. VERİ YÜKLEME
# ════════════════════════════════════════════════════════

def load_data():
    files = [
        ('YARISMA_TRAIN_MASTER.csv','master'),
        ('YARISMA_TRAIN_KANSER.csv','kanser'),
        ('YARISMA_TRAIN_PAH.csv','pah'),
        ('YARISMA_TRAIN_CFTR.csv','cftr'),
    ]
    frames = []
    for path, panel in files:
        try:
            df = pd.read_csv(path, low_memory=False, on_bad_lines='skip')
        except TypeError:
            df = pd.read_csv(path, low_memory=False)
        df['panel'] = panel
        for col in df.columns:
            if df[col].dtype == object and col not in ['panel','AA_1','AA_2']:
                conv = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    if conv.notna().sum()/df[col].notna().sum() >= 0.5:
                        df[col] = conv
        df['Label'] = pd.to_numeric(df['Label'], errors='coerce')
        df = df[df['Label'].notna()].reset_index(drop=True)
        print(f"  [{panel.upper():8s}] {len(df):5d} satır | "
              f"1={df['Label'].sum():.0f} 0={(df['Label']==0).sum():.0f}")
        frames.append(df)
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\n[VERİ] Toplam {len(df_all)} satır")
    return df_all

# ════════════════════════════════════════════════════════
# 5. ÖN İŞLEME — yapısal özellikler eklendi
# ════════════════════════════════════════════════════════

def preprocess(df):
    y       = df['Label'].astype(int).reset_index(drop=True)
    panels  = df['panel'].reset_index(drop=True)
    pan_idx = panels.map(PANEL2IDX).astype(int)

    cat_cols, ohe_dict, cat_frames = [c for c in df.columns if c.startswith('CAT_')], {}, []
    for col in cat_cols:
        s   = df[col].fillna('MISSING').astype(str)
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', min_frequency=3)
        enc = ohe.fit_transform(s.values.reshape(-1,1))
        cat_frames.append(pd.DataFrame(enc,
            columns=[f'{col}_{x}' for x in ohe.get_feature_names_out()]))
        ohe_dict[col] = ohe
    df_cat = pd.concat(cat_frames, axis=1).reset_index(drop=True) if cat_frames else pd.DataFrame()

    aa1_ser = df['AA_1'].fillna('X').astype(str).str.strip().str.upper()
    aa2_ser = df['AA_2'].fillna('X').astype(str).str.strip().str.upper()

    # Biyokimyasal AA özellikleri
    df_aa = pd.DataFrame(
        np.vstack([aa_features(a,b) for a,b in zip(aa1_ser, aa2_ser)]),
        columns=AA_FEAT_NAMES)

    # YENİ: Yapısal özellikler
    df_struct = pd.DataFrame(
        np.vstack([structural_features(a,b) for a,b in zip(aa1_ser, aa2_ser)]),
        columns=STRUCT_FEAT_NAMES)

    # AA özelliklerini yapısal özelliklerle birleştir
    df_aa = pd.concat([df_aa, df_struct], axis=1)

    drop = set(cat_cols + ['AA_1','AA_2','Label','panel','Variant_ID'])
    num_cols = [c for c in df.columns if c not in drop]
    X_raw = df[num_cols].copy().reset_index(drop=True)
    for col in X_raw.columns.tolist():
        if X_raw[col].dtype == object:
            conv = pd.to_numeric(X_raw[col], errors='coerce')
            if X_raw[col].notna().sum()>0 and \
               conv.notna().sum()/X_raw[col].notna().sum()>=0.3:
                X_raw[col] = conv
            else:
                X_raw.drop(columns=[col], inplace=True)

    print(f"[ÖN] Sayısal: {X_raw.shape[1]} | Eksik: {X_raw.isna().mean().mean()*100:.1f}%")

    prof = X_raw.round(3).astype(str).apply('|'.join, axis=1)
    conflict = (pd.DataFrame({'p':prof,'l':y}).groupby('p')['l']
                .transform('nunique')>1).values
    print(f"[KALİTE] Çelişkili: {conflict.sum()}")

    miss_flags = X_raw.isna().astype(int)
    miss_flags.columns = [f'mis_{c}' for c in X_raw.columns]
    mis_mi = mutual_info_classif(miss_flags, y, random_state=RANDOM_SEED)
    df_miss = miss_flags[miss_flags.columns[mis_mi > 0.001]].reset_index(drop=True)
    print(f"[EKSİKLİK] {df_miss.shape[1]} anlamlı gösterge")

    print("[TDA] Persistent homology hesaplanıyor (genisletilmis)...")
    X_for_tda = X_raw.fillna(X_raw.median())
    tda_rows = []
    for i, row in enumerate(X_for_tda.values):
        if (i+1) % 500 == 0: print(f"  {i+1}/{len(X_for_tda)}")
        tda_rows.append(tda_features(row))
    df_tda = pd.DataFrame(tda_rows, columns=TDA_NAMES)
    print(f"[TDA] Tamamlandı — {len(TDA_NAMES)} özellik")

    return (X_raw, df_aa, df_miss, df_cat, df_tda,
            y, panels, pan_idx, pd.Series(conflict), ohe_dict,
            aa1_ser, aa2_ser)

# ════════════════════════════════════════════════════════
# 6. FOLD İÇİ YARDIMCI
# ════════════════════════════════════════════════════════

def _build_fold_features(X_raw_tr, X_raw_val,
                          df_aa, df_miss, df_cat, df_tda,
                          tr_idx, val_idx):
    imputer = IterativeImputer(random_state=RANDOM_SEED, max_iter=7,
                                n_nearest_features=20, sample_posterior=True)
    X_tr_imp = imputer.fit_transform(X_raw_tr)
    X_val_imp = imputer.transform(X_raw_val)

    sc_num = RobustScaler()
    X_tr_sc  = sc_num.fit_transform(X_tr_imp)
    X_val_sc = sc_num.transform(X_val_imp)

    sc_aa = RobustScaler()
    aa_tr  = sc_aa.fit_transform(df_aa.iloc[tr_idx].values)
    aa_val = sc_aa.transform(df_aa.iloc[val_idx].values)

    sc_tda = RobustScaler()
    tda_tr  = sc_tda.fit_transform(df_tda.iloc[tr_idx].values)
    tda_val = sc_tda.transform(df_tda.iloc[val_idx].values)

    miss_tr  = df_miss.iloc[tr_idx].values.astype(np.float32)
    miss_val = df_miss.iloc[val_idx].values.astype(np.float32)
    cat_tr   = df_cat.iloc[tr_idx].values.astype(np.float32)
    cat_val  = df_cat.iloc[val_idx].values.astype(np.float32)

    Xtr  = np.hstack([X_tr_sc,  aa_tr,  tda_tr,  miss_tr,  cat_tr]).astype(np.float32)
    Xval = np.hstack([X_val_sc, aa_val, tda_val, miss_val, cat_val]).astype(np.float32)
    return Xtr, Xval

# ════════════════════════════════════════════════════════
# 7. FT-TRANSFORMER
# ════════════════════════════════════════════════════════

class FeatureTokenizer(nn.Module):
    def __init__(self, n, d):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n, d))
        self.b = nn.Parameter(torch.zeros(n, d))
        nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))
    def forward(self, x):
        return x.unsqueeze(-1)*self.W.unsqueeze(0)+self.b.unsqueeze(0)

class MTLFTTransformer(nn.Module):
    def __init__(self, n_features, d_token=D_TOKEN, n_heads=N_HEADS,
                 n_layers=N_LAYERS, dropout=0.3, n_panels=4):
        super().__init__()
        assert d_token % n_heads == 0
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls = nn.Parameter(torch.zeros(1,1,d_token))
        enc = nn.TransformerEncoderLayer(d_model=d_token, nhead=n_heads,
            dim_feedforward=d_token*4, dropout=dropout,
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.patho_head = nn.Sequential(
            nn.LayerNorm(d_token), nn.Dropout(dropout), nn.Linear(d_token,1))
        self.panel_head = nn.Sequential(
            nn.LayerNorm(d_token), nn.Dropout(dropout), nn.Linear(d_token,n_panels))

    def forward(self, x):
        tok = self.tokenizer(x)
        cls = self.cls.expand(x.size(0),-1,-1)
        out = self.transformer(torch.cat([cls,tok],dim=1))[:,0]
        return self.patho_head(out).squeeze(-1), self.panel_head(out)

    @torch.no_grad()
    def predict_proba(self, X_np, bs=256):
        self.eval()
        probs = []
        t = torch.tensor(X_np, dtype=torch.float32)
        for i in range(0, len(t), bs):
            logits, _ = self.forward(t[i:i+bs].to(DEVICE))
            probs.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(probs)

def train_ft(X_tr, y_tr, pan_tr, X_val, y_val, pos_weight):
    mdl = MTLFTTransformer(X_tr.shape[1]).to(DEVICE)
    pw  = torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    ce  = nn.CrossEntropyLoss()
    opt = optim.AdamW(mdl.parameters(), lr=FT_LR, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
    ds  = TensorDataset(torch.tensor(X_tr,dtype=torch.float32),
                        torch.tensor(y_tr,dtype=torch.float32),
                        torch.tensor(pan_tr,dtype=torch.long))
    ldr = DataLoader(ds, batch_size=FT_BATCH, shuffle=True)
    best_auc, best_state = 0., None
    for _ in range(FT_EPOCHS):
        mdl.train()
        for xb,yb,pb in ldr:
            xb,yb,pb = xb.to(DEVICE),yb.to(DEVICE),pb.to(DEVICE)
            yb_smooth = yb*0.9+0.05
            p_logit, pan_logit = mdl(xb)
            loss = MTL_ALPHA*bce(p_logit,yb_smooth)+(1-MTL_ALPHA)*ce(pan_logit,pb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(mdl.parameters(),1.0)
            opt.step()
        sch.step()
        val_auc = roc_auc_score(y_val, mdl.predict_proba(X_val))
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.clone() for k,v in mdl.state_dict().items()}
    mdl.load_state_dict(best_state)
    print(f"  [MTL-FT] Val AUC: {best_auc:.4f}")
    return mdl

# ════════════════════════════════════════════════════════
# 8. EŞİK & METRİKLER
# ════════════════════════════════════════════════════════

def best_threshold(y_true, y_proba, min_spec=MIN_SPEC):
    _, _, thrs = precision_recall_curve(y_true, y_proba)
    best_f2, best_t = 0., 0.5
    for t in thrs:
        yp = (y_proba >= t).astype(int)
        tn,fp,fn,tp = confusion_matrix(y_true,yp,labels=[0,1]).ravel()
        if tn/(tn+fp+1e-12) < min_spec: continue
        f2 = fbeta_score(y_true,yp,beta=2,zero_division=0)
        if f2 > best_f2: best_f2, best_t = f2, float(t)
    return float(best_t), float(best_f2)

def metrics(y_true, y_proba, thr):
    yp = (y_proba >= thr).astype(int)
    tn,fp,fn,tp = confusion_matrix(y_true,yp,labels=[0,1]).ravel()
    return {
        'roc_auc':roc_auc_score(y_true,y_proba),
        'pr_auc':average_precision_score(y_true,y_proba),
        'f1':f1_score(y_true,yp,zero_division=0),
        'f2':fbeta_score(y_true,yp,beta=2,zero_division=0),
        'recall':tp/(tp+fn+1e-12), 'spec':tn/(tn+fp+1e-12),
        'mcc':matthews_corrcoef(y_true,yp),
        'bal_acc':balanced_accuracy_score(y_true,yp),
        'accuracy':(tp+tn)/(tp+tn+fp+fn), 'threshold':thr,
        'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
    }

def print_metrics(m, label=''):
    print(f"\n--- {label} ---")
    for k in ['roc_auc','pr_auc','f1','f2','recall','spec','mcc','bal_acc','accuracy','threshold']:
        print(f"  {k:<12}: {m[k]:.4f}")
    print(f"  CM: TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}")

# ════════════════════════════════════════════════════════
# YENİ: BOOTSTRAP GÜVEN ARALIKLARI
# ════════════════════════════════════════════════════════

def bootstrap_ci(y_true, y_proba, thr, n_boot=N_BOOT, alpha=0.05, seed=RANDOM_SEED):
    """
    ROC-AUC, PR-AUC, F2, MCC için %95 bootstrap güven aralığı.
    Returns dict: metric -> (mean, lower, upper)
    """
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    boot_metrics = {'roc_auc':[], 'pr_auc':[], 'f2':[], 'mcc':[]}

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt  = y_true[idx]; yp = y_proba[idx]
        if len(np.unique(yt)) < 2: continue
        yb = (yp >= thr).astype(int)
        boot_metrics['roc_auc'].append(roc_auc_score(yt, yp))
        boot_metrics['pr_auc'].append(average_precision_score(yt, yp))
        boot_metrics['f2'].append(fbeta_score(yt, yb, beta=2, zero_division=0))
        boot_metrics['mcc'].append(matthews_corrcoef(yt, yb))

    result = {}
    for key, vals in boot_metrics.items():
        arr = np.array(vals)
        result[key] = (
            float(np.mean(arr)),
            float(np.percentile(arr, 100*alpha/2)),
            float(np.percentile(arr, 100*(1-alpha/2)))
        )
    return result

def print_bootstrap_ci(ci_dict, label='Bootstrap %95 GA'):
    print(f"\n[{label}]")
    for k,(mean,lo,hi) in ci_dict.items():
        print(f"  {k:<12}: {mean:.4f}  [{lo:.4f} – {hi:.4f}]")

# ════════════════════════════════════════════════════════
# YENİ: VARYANT ETKİLEŞİM AĞI
# ════════════════════════════════════════════════════════

def plot_variant_network(df, oof_stack, panels, thr_stack,
                          n_sample=200, save_path='outputs/variant_network.png'):
    """
    Varyantlar arası benzerlik ağı.
    Düğümler: varyantlar (renk=panel, boyut=patojenik olasılık)
    Kenarlar: Grantham benzerliği yüksek varyant çiftleri
    """
    if not NX_OK:
        print("[NETWORK] networkx yok, atlandı")
        return

    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(df), min(n_sample, len(df)), replace=False)

    aa1s = df['AA_1'].iloc[idx].fillna('X').str.upper().values
    aa2s = df['AA_2'].iloc[idx].fillna('X').str.upper().values
    pans = panels.iloc[idx].values
    prbs = oof_stack[idx]
    labs = (prbs >= thr_stack).astype(int)

    G = nx.Graph()
    for i, (a1,a2,pan,prb,lab) in enumerate(zip(aa1s,aa2s,pans,prbs,labs)):
        G.add_node(i, panel=pan, proba=prb, label=lab, aa1=a1, aa2=a2)

    # Kenar: aynı amino asit değişimine sahip varyantlar
    for i in range(len(idx)):
        for j in range(i+1, len(idx)):
            g1 = GRANTHAM.get((aa1s[i], aa2s[i]), 0)
            g2 = GRANTHAM.get((aa1s[j], aa2s[j]), 0)
            if abs(g1-g2) < 20 and pans[i] == pans[j]:
                G.add_edge(i, j, weight=1/(abs(g1-g2)+1))

    # Görselleştirme
    fig, ax = plt.subplots(figsize=(12,9))
    panel_colors = {'master':'#4C72B0','kanser':'#DD8452',
                    'pah':'#55A868','cftr':'#C44E52'}
    node_colors = [panel_colors.get(G.nodes[n]['panel'],'gray') for n in G.nodes]
    node_sizes  = [200 + 300*G.nodes[n]['proba'] for n in G.nodes]
    node_alpha  = [0.9 if G.nodes[n]['label']==1 else 0.4 for n in G.nodes]

    pos = nx.spring_layout(G, k=0.5, seed=RANDOM_SEED)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                           node_size=node_sizes, alpha=0.7, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.2, edge_color='gray', ax=ax)

    from matplotlib.patches import Patch
    legend = [Patch(color=c, label=p.upper()) for p,c in panel_colors.items()]
    ax.legend(handles=legend, loc='upper left', fontsize=9)
    ax.set_title('Varyant Etkilesim Agi\n(Dugum boyutu ~ patojenik olasilik, renk = panel)',
                 fontsize=12)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[NETWORK] {save_path} — {G.number_of_nodes()} dugum, {G.number_of_edges()} kenar")

# ════════════════════════════════════════════════════════
# 9. ANA EĞİTİM
# ════════════════════════════════════════════════════════

def train_all(X_raw, df_aa, df_miss, df_cat, df_tda,
              y, panels, pan_idx, conflict_mask):

    sgkf   = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True,
                                   random_state=RANDOM_SEED)
    groups = pd.util.hash_pandas_object(X_raw, index=False) % 200
    conflict_array = conflict_mask.values.flatten()

    pw = float((y==0).sum()/(y==1).sum()) * 1.5
    n  = len(y)
    oof_ft  = np.zeros(n)
    oof_xgb = np.zeros(n)
    oof_lgb = np.zeros(n)
    fold_results, models_ft, models_xgb, models_lgb = [], [], [], []

    for fold, (tr_idx, val_idx) in enumerate(sgkf.split(X_raw, y, groups)):
        print(f"\n{'='*50}\nFold {fold+1}/{N_FOLDS}")

        clean = [i for i in tr_idx if not conflict_array[i]]
        ytr   = y.iloc[clean].values
        yval  = y.iloc[val_idx].values
        pan_tr = pan_idx.iloc[clean].values

        print("  [MICE + Scale] çalışıyor...")
        Xtr_np, Xval_np = _build_fold_features(
            X_raw.iloc[clean].values, X_raw.iloc[val_idx].values,
            df_aa, df_miss, df_cat, df_tda, clean, val_idx)
        print(f"  Özellik boyutu: {Xtr_np.shape[1]}")

        n_ornek   = len(ytr)
        n_ozellik = Xtr_np.shape[1]
        oran = n_ornek / n_ozellik
        max_depth_ust = 4 if oran < 1 else (5 if oran < 3 else 9)

        cftr_ratio = (pan_tr == PANEL2IDX['cftr']).mean()
        if cftr_ratio > 0.01:
            max_depth_ust = min(max_depth_ust, 3)

        def xgb_objective(trial):
            p = {
                'n_estimators'    : trial.suggest_int('n_est', 100, 700),
                'learning_rate'   : trial.suggest_float('lr', 0.01, 0.1, log=True),
                'max_depth'       : trial.suggest_int('md', 3, max_depth_ust),
                'subsample'       : trial.suggest_float('sub', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('col', 0.6, 1.0),
                'reg_alpha'       : trial.suggest_float('a', 0.1, 50, log=True),
                'reg_lambda'      : trial.suggest_float('l', 0.1, 50, log=True),
                'tree_method':'hist', 'scale_pos_weight':pw,
                'random_state':RANDOM_SEED, 'n_jobs':-1,
            }
            m = xgb.XGBClassifier(**p)
            m.fit(Xtr_np, ytr, eval_set=[(Xval_np,yval)], verbose=False)
            return log_loss(yval, m.predict_proba(Xval_np)[:,1])

        print("  [XGB Optuna] başlıyor...")
        xgb_study = optuna.create_study(direction='minimize',
                        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
        xgb_study.optimize(xgb_objective, n_trials=OPTUNA_N, show_progress_bar=False)
        best_xgb_p = xgb_study.best_params
        best_xgb_p.update({'tree_method':'hist','scale_pos_weight':pw,
                           'random_state':RANDOM_SEED,'n_jobs':-1})
        print(f"  [XGB] best log_loss: {xgb_study.best_value:.4f}")

        def lgb_objective(trial):
            p = {
                'n_estimators'    : trial.suggest_int('n_est', 100, 700),
                'learning_rate'   : trial.suggest_float('lr', 0.01, 0.1, log=True),
                'max_depth'       : trial.suggest_int('md', 3, max_depth_ust),
                'num_leaves'      : trial.suggest_int('nl', 15, 127),
                'subsample'       : trial.suggest_float('sub', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('col', 0.6, 1.0),
                'reg_alpha'       : trial.suggest_float('a', 0.1, 50, log=True),
                'reg_lambda'      : trial.suggest_float('l', 0.1, 50, log=True),
                'scale_pos_weight':pw, 'random_state':RANDOM_SEED,
                'verbose':-1, 'n_jobs':-1,
            }
            m = lgb.LGBMClassifier(**p)
            m.fit(Xtr_np, ytr, eval_set=[(Xval_np,yval)],
                  callbacks=[lgb.early_stopping(30,verbose=False)])
            return log_loss(yval, m.predict_proba(Xval_np)[:,1])

        print("  [LGB Optuna] başlıyor...")
        lgb_study = optuna.create_study(direction='minimize',
                        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
        lgb_study.optimize(lgb_objective, n_trials=OPTUNA_N, show_progress_bar=False)
        best_lgb_p = lgb_study.best_params
        best_lgb_p.update({'scale_pos_weight':pw,'random_state':RANDOM_SEED,
                           'verbose':-1,'n_jobs':-1})
        print(f"  [LGB] best log_loss: {lgb_study.best_value:.4f}")

        ft = train_ft(Xtr_np, ytr, pan_tr, Xval_np, yval, pw)
        oof_ft[val_idx] = ft.predict_proba(Xval_np)
        models_ft.append(ft)

        m_xgb = xgb.XGBClassifier(**best_xgb_p)
        m_xgb.fit(Xtr_np, ytr, eval_set=[(Xval_np,yval)], verbose=False)
        p_xgb = m_xgb.predict_proba(Xval_np)[:,1]
        oof_xgb[val_idx] = p_xgb
        models_xgb.append(m_xgb)

        m_lgb = lgb.LGBMClassifier(**best_lgb_p)
        m_lgb.fit(Xtr_np, ytr, eval_set=[(Xval_np,yval)],
                  callbacks=[lgb.early_stopping(50,verbose=False),
                              lgb.log_evaluation(-1)])
        p_lgb = m_lgb.predict_proba(Xval_np)[:,1]
        oof_lgb[val_idx] = p_lgb
        models_lgb.append(m_lgb)

        p_ens = (oof_ft[val_idx] + p_xgb + p_lgb) / 3
        thr, _ = best_threshold(yval, p_ens)
        m = metrics(yval, p_ens, thr)
        print_metrics(m, f"Fold {fold+1} (basit ens)")

        pv = panels.iloc[val_idx]; pm = {}
        for pname in pv.unique():
            msk = (pv == pname).values

            if msk.sum() < 5: continue
            pt, _ = best_threshold(yval[msk], p_ens[msk])
            pm[pname] = metrics(yval[msk], p_ens[msk], pt)

            auc = pm[pname]['roc_auc']
            flag = " ⚠ (n=%d, güvenilmez)" % msk.sum() if auc >= 0.999 else ""
            print(f"  [{pname.upper()}] AUC={auc:.4f} "
                  f"F2={pm[pname]['f2']:.4f} MCC={pm[pname]['mcc']:.4f}{flag}")
        fold_results.append({'fold':fold+1,'metrics':m,'panel_metrics':pm,
                             'val_idx':val_idx})

    print("\n[STACKING] Meta-öğrenici eğitiliyor...")
    meta_features = np.column_stack([oof_ft, oof_xgb, oof_lgb])
    meta = LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_SEED)
    meta.fit(meta_features, y.values)
    oof_stack = meta.predict_proba(meta_features)[:,1]

    thr_stack, _ = best_threshold(y.values, oof_stack)
    m_stack = metrics(y.values, oof_stack, thr_stack)
    print_metrics(m_stack, "STACKING OOF")

    panel_stack = {}
    for pname in panels.unique():
        msk = (panels == pname).values
        if msk.sum() < 5: continue
        pt, _ = best_threshold(y.values[msk], oof_stack[msk])
        panel_stack[pname] = metrics(y.values[msk], oof_stack[msk], pt)
        print(f"  [{pname.upper()}] AUC={panel_stack[pname]['roc_auc']:.4f} "
              f"F2={panel_stack[pname]['f2']:.4f} MCC={panel_stack[pname]['mcc']:.4f}")

    return (fold_results, models_ft, models_xgb, models_lgb, meta,
            oof_ft, oof_xgb, oof_lgb, oof_stack, thr_stack, m_stack, panel_stack)

# ════════════════════════════════════════════════════════
# 10. PANEL FINE-TUNING
# ════════════════════════════════════════════════════════

def panel_fine_tune(X_raw, df_aa, df_miss, df_cat, df_tda,
                    y, panels, fold_results, models_xgb):
    print("\n[FINE-TUNE] Panel-spesifik ince ayar başlıyor...")

    def best_fold_for_panel(pname):   # ← 4 boşluk
        scores = []
        for f in fold_results:
            pm = f.get('panel_metrics', {})
            score = pm.get(pname, {}).get('roc_auc', 0)
            if score >= 0.999:
                score = 0
            scores.append(score)
        best = int(np.argmax(scores))
        if max(scores) == 0:
            best = int(np.argmax([f['metrics']['pr_auc'] for f in fold_results]))
        return best


    ft_models = {}
    for pname in PANELS:
        best_fold_idx = best_fold_for_panel(pname)   # ← YENİ
        base_model    = models_xgb[best_fold_idx]    # ← YENİ
        print(f"  Base model: Fold {best_fold_idx+1} ({pname} için)")
        msk = (panels == pname).values
        if msk.sum() < 20:
            print(f"  [{pname.upper()}] Yetersiz veri, atlandı")
            continue

        idx_all = np.where(msk)[0]
        y_p = y.iloc[idx_all].reset_index(drop=True)

        tr_rel, val_rel = train_test_split(
            range(len(idx_all)), test_size=0.20,
            random_state=RANDOM_SEED, stratify=y_p)
        tr_abs  = idx_all[list(tr_rel)]
        val_abs = idx_all[list(val_rel)]

        Xp_tr, Xp_val = _build_fold_features(
            X_raw.iloc[tr_abs].values, X_raw.iloc[val_abs].values,
            df_aa, df_miss, df_cat, df_tda, tr_abs, val_abs)

        yp_tr  = y.iloc[tr_abs].values
        yp_val = y.iloc[val_abs].values

        n_ornek   = len(yp_tr)
        n_ozellik = Xp_tr.shape[1]
        oran = n_ornek / n_ozellik
        if oran < 1:   max_depth = 3
        elif oran < 3: max_depth = 4
        else:          max_depth = 6

        if pname == 'pah':
            max_depth = max(max_depth, 4)

        if pname == 'cftr':  # ← 8 boşluk, DOĞRU
            shap_exp = shap.TreeExplainer(base_model)
            shap_importance = np.abs(shap_exp.shap_values(Xp_tr)).mean(axis=0)
            top50_idx = np.argsort(shap_importance)[-50:]
            Xp_tr_fit  = Xp_tr[:, top50_idx]
            Xp_val_fit = Xp_val[:, top50_idx]
            p = {
                'n_estimators'    : 30,
                'max_depth'       : 2,
                'reg_alpha'       : 5.0,
                'reg_lambda'      : 5.0,
                'subsample'       : 0.7,
                'colsample_bytree': 0.5,
                'scale_pos_weight': float((yp_tr==0).sum()/(yp_tr==1).sum()+1e-5),
                'random_state'    : RANDOM_SEED,
                'tree_method'     : 'hist',
            }
            m = xgb.XGBClassifier(**p)
            m.fit(Xp_tr_fit, yp_tr)  # xgb_model yok — farklı boyut
        else:
            Xp_tr_fit  = Xp_tr
            Xp_val_fit = Xp_val
            n_trees = 200 if pname == 'pah' else 50
            p = {
                'n_estimators'    : n_trees,
                'max_depth'       : max_depth,
                'scale_pos_weight': float((yp_tr==0).sum()/(yp_tr==1).sum()+1e-5),
                'random_state'    : RANDOM_SEED,
                'tree_method'     : 'hist',
            }
            m = xgb.XGBClassifier(**p)
            m.fit(Xp_tr, yp_tr, xgb_model=base_model.get_booster())
        ft_models[pname] = m

        proba = m.predict_proba(Xp_val_fit)[:,1]
        thr, _ = best_threshold(yp_val, proba)
        met = metrics(yp_val, proba, thr)
        actual_depth = p.get('max_depth', max_depth)
        print(f"  [{pname.upper()}] Val AUC={met['roc_auc']:.4f} "
              f"F2={met['f2']:.4f} MCC={met['mcc']:.4f} depth={actual_depth}")

    return ft_models

# ════════════════════════════════════════════════════════
# 11. İSTATİSTİKSEL TESTLER
# ════════════════════════════════════════════════════════

def delong_test(y_true, pa, pb):
    aa = roc_auc_score(y_true,pa); ab = roc_auc_score(y_true,pb)
    pp_a,np_a = pa[y_true==1],pa[y_true==0]
    pp_b,np_b = pb[y_true==1],pb[y_true==0]
    n1,n0 = len(pp_a),len(np_a)
    V10_a=np.array([np.mean(p>np_a)+.5*np.mean(p==np_a) for p in pp_a])
    V01_a=np.array([np.mean(p<pp_a)+.5*np.mean(p==pp_a) for p in np_a])
    V10_b=np.array([np.mean(p>np_b)+.5*np.mean(p==np_b) for p in pp_b])
    V01_b=np.array([np.mean(p<pp_b)+.5*np.mean(p==pp_b) for p in np_b])
    S10=np.cov(V10_a,V10_b)/n1 if n1>1 else np.zeros((2,2))
    S01=np.cov(V01_a,V01_b)/n0 if n0>1 else np.zeros((2,2))
    S=S10+S01; diff=aa-ab; var=S[0,0]+S[1,1]-2*S[0,1]
    if var<=0: return aa,ab,diff,1.0
    p=2*(1-stats.norm.cdf(abs(diff/np.sqrt(var))))
    return aa,ab,diff,float(p)

def mcnemar_test(y_true, pa, pb):
    b=np.sum((pa!=y_true)&(pb==y_true)); c=np.sum((pa==y_true)&(pb!=y_true))
    if (b+c)==0: return 0.,1.
    chi2=(abs(b-c)-1)**2/(b+c)
    return float(chi2),float(1-stats.chi2.cdf(chi2,df=1))

def run_statistical_tests(y_true, oof_xgb, oof_lgb, oof_stack,
                           thr_xgb, thr_lgb, thr_stack):
    print("\n[İSTATİSTİK] DeLong ve McNemar Testleri")
    bonferroni = 0.05/3
    pairs = [("XGB vs LGB",oof_xgb,oof_lgb,thr_xgb,thr_lgb),
             ("XGB vs Stack",oof_xgb,oof_stack,thr_xgb,thr_stack),
             ("LGB vs Stack",oof_lgb,oof_stack,thr_lgb,thr_stack)]
    results = []
    print(f"\n{'Karşılaştırma':<20} {'AUC_A':>7} {'AUC_B':>7} "
          f"{'Fark':>7} {'DeLong_p':>10} {'McNemar_p':>10} {'Anlamlı?':>10}")
    for name,pa,pb,ta,tb in pairs:
        aa,ab,diff,pd_ = delong_test(y_true,pa,pb)
        _,pm = mcnemar_test(y_true,(pa>=ta).astype(int),(pb>=tb).astype(int))
        sig = "✓" if pd_<bonferroni else "—"
        print(f"  {name:<20} {aa:>7.4f} {ab:>7.4f} {diff:>+7.4f} "
              f"{pd_:>10.4f} {pm:>10.4f} {sig:>10}")
        results.append({'name':name,'auc_a':aa,'auc_b':ab,
                        'diff':diff,'p_delong':pd_,'p_mcn':pm})
    print(f"\n  Bonferroni eşiği: {bonferroni:.4f}")
    return results

# ════════════════════════════════════════════════════════
# 12. DCA & KALİBRASYON
# ════════════════════════════════════════════════════════

def decision_curve_analysis(y_true, probas_dict, thr_range=(0.05,0.20),
                              save_path='dca.png'):
    thrs=np.linspace(thr_range[0],thr_range[1],100); n=len(y_true)
    fig,ax=plt.subplots(figsize=(9,6))
    ax.plot(thrs,[y_true.sum()/n-(1-y_true).sum()/n*(e/(1-e+1e-9))
                  for e in thrs],'k--',lw=1.5,label='Herkese Patojenik De')
    ax.axhline(0,color='gray',lw=1,label='Kimseye Deme')
    for (name,proba),col in zip(probas_dict.items(),
                                 ['#1f77b4','#ff7f0e','#2ca02c','#d62728']):
        nbs=[]
        for pt in thrs:
            yp=(proba>=pt).astype(int)
            tn,fp,fn,tp_v=confusion_matrix(y_true,yp,labels=[0,1]).ravel()
            nbs.append(tp_v/n-fp/n*(pt/(1-pt+1e-9)))
        ax.plot(thrs,nbs,lw=2.5,color=col,label=name)
    ax.set_xlabel('Karar Eşiği'); ax.set_ylabel('Net Fayda')
    ax.set_title('Decision Curve Analysis')
    ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(thr_range)
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close(); print(f"[DCA] {save_path}")

def calibration_analysis(y_true, y_proba_raw, name='Model',
                          save_path='calibration.png'):
    from sklearn.isotonic import IsotonicRegression
    cal_size = int(len(y_true)*0.30)
    iso = IsotonicRegression(out_of_bounds='clip')
    cal_idx = np.random.choice(len(y_true), cal_size, replace=False)
    iso.fit(y_proba_raw[cal_idx], y_true[cal_idx])
    y_cal = iso.predict(y_proba_raw)
    def ece(y,p,nb=10):
        s=np.linspace(0,1,nb+1); e=0.
        for i in range(nb):
            mk=(p>=s[i])&(p<s[i+1])
            if mk.sum()==0: continue
            e+=mk.sum()/len(y)*abs(y[mk].mean()-p[mk].mean())
        return e
    print(f"\n[KALİBRASYON] {name}")
    print(f"  Brier — Ham: {brier_score_loss(y_true,y_proba_raw):.4f} | "
          f"Kalibre: {brier_score_loss(y_true,y_cal):.4f}")
    print(f"  ECE   — Ham: {ece(y_true,y_proba_raw):.4f} | "
          f"Kalibre: {ece(y_true,y_cal):.4f}")
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,p,t,c in [(axes[0],y_proba_raw,f'Ham ({name})','#1f77b4'),
                     (axes[1],y_cal,f'Kalibre ({name})','#2ca02c')]:
        fp,mp=calibration_curve(y_true,p,n_bins=10)
        ax.plot([0,1],[0,1],'k--',lw=1.5); ax.plot(mp,fp,'o-',color=c,lw=2.5,label=t)
        ax.fill_between(mp,fp,mp,alpha=0.15,color=c)
        ax.set_title(t); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle('Kalibrasyon Analizi'); plt.tight_layout()
    plt.savefig(save_path,dpi=150,bbox_inches='tight'); plt.close()
    print(f"[KALİBRASYON] {save_path}")
    return y_cal

# ════════════════════════════════════════════════════════
# 13. GÖRSELLEŞTİRME
# ════════════════════════════════════════════════════════

def plot_roc_pr(y_true, probas_dict, save_path='roc_pr.png'):
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for (name,p),col in zip(probas_dict.items(),
                             ['#1f77b4','#ff7f0e','#2ca02c','#d62728']):
        fpr,tpr,_=roc_curve(y_true,p)
        axes[0].plot(fpr,tpr,lw=2,color=col,
                     label=f'{name} AUC={roc_auc_score(y_true,p):.4f}')
        prec,rec,_=precision_recall_curve(y_true,p)
        axes[1].plot(rec,prec,lw=2,color=col,
                     label=f'{name} PR={average_precision_score(y_true,p):.4f}')
    axes[0].plot([0,1],[0,1],'k--',lw=1)
    axes[0].set_title('ROC Eğrisi'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_title('PR Eğrisi'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close(); print(f"[GRAFIK] {save_path}")

def plot_shap(model_xgb, X_sample, save_path='shap_summary.png'):
    exp = shap.TreeExplainer(model_xgb)
    sv  = exp.shap_values(X_sample)
    plt.figure(figsize=(10,8))
    shap.summary_plot(sv, X_sample, show=False, max_display=20)
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close(); print(f"[SHAP] {save_path}")

def plot_threshold_scan(y_true, y_proba, optimal_thr, save_path='threshold.png'):
    thrs=np.linspace(0.01,0.99,200); f1s,f2s,recs,specs=[],[],[],[]
    for t in thrs:
        yp=(y_proba>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y_true,yp,labels=[0,1]).ravel()
        f1s.append(f1_score(y_true,yp,zero_division=0))
        f2s.append(fbeta_score(y_true,yp,beta=2,zero_division=0))
        recs.append(tp/(tp+fn+1e-12)); specs.append(tn/(tn+fp+1e-12))
    plt.figure(figsize=(10,5))
    for vals,lbl,ls in [(f1s,'F1','-'),(f2s,'F2','--'),
                        (recs,'Recall',':'),(specs,'Specificity','-.')]:
        plt.plot(thrs,vals,lw=2,label=lbl,linestyle=ls)
    plt.axvline(optimal_thr,color='red',lw=2,linestyle='--',
                label=f'Optimal={optimal_thr:.3f}')
    plt.xlabel('Karar Eşiği'); plt.ylabel('Skor')
    plt.title('Eşik Taraması'); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close(); print(f"[GRAFIK] {save_path}")

def plot_panel_comparison(panel_metrics, save_path='panels.png'):
    pnames=[p for p in PANELS if p in panel_metrics]
    fig,axes=plt.subplots(1,4,figsize=(16,5))
    for ax,m,lbl,col in zip(axes,
        ['roc_auc','pr_auc','f2','mcc'],['ROC-AUC','PR-AUC','F2','MCC'],
        ['#4C72B0','#DD8452','#55A868','#C44E52']):
        vals=[panel_metrics[p][m] for p in pnames]
        ax.bar(pnames,vals,color=col,alpha=0.85)
        ax.set_title(lbl); ax.set_ylim(0,1.1); ax.grid(axis='y',alpha=0.3)
        for i,v in enumerate(vals):
            ax.text(i,v+0.02,f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
    plt.suptitle('Panel Karşılaştırması (Stacking OOF)')
    plt.tight_layout(); plt.savefig(save_path,dpi=150,bbox_inches='tight')
    plt.close(); print(f"[GRAFIK] {save_path}")

# ════════════════════════════════════════════════════════
# YENİ: TEST INFERENCE — yarışma günü kullanılacak
# ════════════════════════════════════════════════════════

def predict_test(test_csv_path, models_xgb, models_lgb, models_ft,
                  meta_learner, thr_stack, ohe_dict, feat_names,
                  X_raw_train, df_miss_cols, df_cat_cols):
    """
    Yarışma günü test verisi geldiğinde çalıştırılacak fonksiyon.
    Eğitim verisiyle aynı ön işleme uygulanır, tahminler CSV'ye kaydedilir.

    Kullanım:
        predict_test('test.csv', models_xgb, models_lgb, models_ft,
                      meta_learner, thr_stack, ohe_dict, feat_names,
                      X_raw, df_miss.columns, df_cat.columns)
    """
    print(f"\n[TEST INFERENCE] {test_csv_path} yükleniyor...")
    try:
        df_test = pd.read_csv(test_csv_path, low_memory=False, on_bad_lines='skip')
    except TypeError:
        df_test = pd.read_csv(test_csv_path, low_memory=False)

    n_test = len(df_test)
    print(f"  {n_test} test varyantı")

    # AA özellikleri
    aa1_t = df_test['AA_1'].fillna('X').astype(str).str.strip().str.upper()
    aa2_t = df_test['AA_2'].fillna('X').astype(str).str.strip().str.upper()
    aa_feats = np.vstack([aa_features(a,b) for a,b in zip(aa1_t,aa2_t)])
    struct_feats = np.vstack([structural_features(a,b) for a,b in zip(aa1_t,aa2_t)])

    # CAT özellikleri — eğitimde fit edilen OHE kullan
    cat_frames = []
    for col, ohe in ohe_dict.items():
        if col in df_test.columns:
            s = df_test[col].fillna('MISSING').astype(str)
            enc = ohe.transform(s.values.reshape(-1,1))
            cat_frames.append(pd.DataFrame(enc,
                columns=[f'{col}_{x}' for x in ohe.get_feature_names_out()]))
    df_cat_test = pd.concat(cat_frames, axis=1) if cat_frames else pd.DataFrame(
        np.zeros((n_test, len(df_cat_cols))), columns=df_cat_cols)

    # Sayısal sütunlar
    drop = set(list(ohe_dict.keys()) + ['AA_1','AA_2','Label','panel','Variant_ID'])
    num_cols_t = [c for c in X_raw_train.columns if c in df_test.columns]
    X_test_raw = df_test[num_cols_t].copy()
    for col in num_cols_t:
        X_test_raw[col] = pd.to_numeric(X_test_raw[col], errors='coerce')

    # Eksiklik göstergesi
    miss_t = X_test_raw.isna().astype(int)
    miss_t.columns = [f'mis_{c}' for c in X_test_raw.columns]
    # Eğitimdeki sütunlarla hizala
    for col in df_miss_cols:
        if col not in miss_t.columns:
            miss_t[col] = 0
    miss_t = miss_t[df_miss_cols]

    # TDA
    print("  [TDA] Test verisi için hesaplanıyor...")
    X_for_tda_t = X_test_raw.fillna(X_test_raw.median())
    tda_rows_t = []
    for i, row in enumerate(X_for_tda_t.values):
        tda_rows_t.append(tda_features(row))
    df_tda_test = pd.DataFrame(tda_rows_t, columns=TDA_NAMES)

    # MICE + Scale
    print("  [MICE] Test verisi impute ediliyor...")
    imputer = IterativeImputer(random_state=RANDOM_SEED, max_iter=7,
                                n_nearest_features=20, sample_posterior=True)
    imputer.fit(X_raw_train.values)
    X_test_imp = imputer.transform(X_test_raw.values)

    sc_num = RobustScaler()
    sc_num.fit(X_raw_train.values)
    X_test_sc = sc_num.transform(X_test_imp)

    sc_aa = RobustScaler()
    sc_aa.fit(np.hstack([aa_feats, struct_feats]))
    aa_test_sc = sc_aa.transform(np.hstack([aa_feats, struct_feats]))

    sc_tda = RobustScaler()
    sc_tda.fit(df_tda_test.values)
    tda_test_sc = sc_tda.transform(df_tda_test.values)

    X_test_full = np.hstack([
        X_test_sc, aa_test_sc, tda_test_sc,
        miss_t.values.astype(np.float32),
        df_cat_test.values.astype(np.float32)
    ]).astype(np.float32)

    # Ensemble tahmin
    print("  [ENSEMBLE] Tahminler üretiliyor...")
    preds_xgb = np.mean([m.predict_proba(X_test_full)[:,1] for m in models_xgb], axis=0)
    preds_lgb = np.mean([m.predict_proba(X_test_full)[:,1] for m in models_lgb], axis=0)
    preds_ft  = np.mean([m.predict_proba(X_test_full) for m in models_ft], axis=0)

    meta_input = np.column_stack([preds_ft, preds_xgb, preds_lgb])
    preds_stack = meta_learner.predict_proba(meta_input)[:,1]

    # Submission
    variant_ids = df_test['Variant_ID'].values if 'Variant_ID' in df_test.columns \
                  else [f'VAR_{i:05d}' for i in range(n_test)]
    submission = pd.DataFrame({
        'Variant_ID' : variant_ids,
        'Label'      : (preds_stack >= thr_stack).astype(int),
        'Probability': preds_stack
    })
    out_path = 'submission.csv'
    submission.to_csv(out_path, index=False)
    print(f"\n[TEST] submission.csv hazır — {n_test} tahmin")
    print(f"  Patojenik: {submission['Label'].sum()} ({submission['Label'].mean()*100:.1f}%)")
    print(f"  Benign   : {(submission['Label']==0).sum()}")
    return submission

# ════════════════════════════════════════════════════════
# 14. ANA AKIŞ
# ════════════════════════════════════════════════════════

if __name__ == '__main__':
    os.makedirs('outputs', exist_ok=True)

    print("\n=== 1. VERİ YÜKLEME ===")
    df = load_data()

    print("\n=== 2. ÖN İŞLEME ===")
    (X_raw, df_aa, df_miss, df_cat, df_tda,
     y, panels, pan_idx, conflict_mask, ohe_dict,
     aa1_ser, aa2_ser) = preprocess(df)

    print("\n=== 3. ANA EĞİTİM + STACKING ===")
    (fold_results, models_ft, models_xgb, models_lgb, meta_learner,
     oof_ft, oof_xgb, oof_lgb, oof_stack, thr_stack,
     m_stack, panel_stack) = train_all(
        X_raw, df_aa, df_miss, df_cat, df_tda,
        y, panels, pan_idx, conflict_mask)

    print("\n=== 4. PANEL FINE-TUNING ===")
    ft_panel_models = panel_fine_tune(
        X_raw, df_aa, df_miss, df_cat, df_tda,
        y, panels, fold_results, models_xgb)

    print("\n=== 5. İSTATİSTİKSEL TESTLER ===")
    thr_xgb, _ = best_threshold(y.values, oof_xgb)
    thr_lgb, _ = best_threshold(y.values, oof_lgb)
    stat_results = run_statistical_tests(
        y.values, oof_xgb, oof_lgb, oof_stack,
        thr_xgb, thr_lgb, thr_stack)

    print("\n=== 5b. BOOTSTRAP GÜVEN ARALIKLARI ===")
    ci = bootstrap_ci(y.values, oof_stack, thr_stack, n_boot=N_BOOT)
    print_bootstrap_ci(ci)

    print("\n=== 6. KALİBRASYON ===")
    calibration_analysis(y.values, oof_stack, name='Stacking',
                          save_path='outputs/calibration.png')

    print("\n=== 7. DCA ===")
    decision_curve_analysis(y.values,
        {'XGBoost':oof_xgb,'LightGBM':oof_lgb,'Stacking':oof_stack},
        save_path='outputs/dca.png')

    print("\n=== 8. GÖRSELLEŞTİRME ===")
    plot_roc_pr(y.values,
        {'XGBoost':oof_xgb,'LightGBM':oof_lgb,'Stacking':oof_stack},
        save_path='outputs/roc_pr.png')
    plot_threshold_scan(y.values, oof_stack, thr_stack,
                         save_path='outputs/threshold.png')
    plot_panel_comparison(panel_stack, save_path='outputs/panels.png')

    # Varyant Etkileşim Ağı
    print("\n=== 8b. VARYANT ETKİLEŞİM AĞI ===")
    plot_variant_network(df, oof_stack, panels, thr_stack,
                          save_path='outputs/variant_network.png')

    # Özellik isimleri
    feat_names = (list(X_raw.columns) +
                  AA_FEAT_NAMES + STRUCT_FEAT_NAMES +
                  TDA_NAMES +
                  list(df_miss.columns) +
                  list(df_cat.columns))

    # SHAP
    best_fold_idx = np.argmax([f['metrics']['pr_auc'] for f in fold_results])
    val_idx_shap  = fold_results[best_fold_idx]['val_idx']
    clean_shap    = [i for i in range(len(y)) if i not in set(val_idx_shap)]
    Xtr_shap, Xval_shap = _build_fold_features(
        X_raw.iloc[clean_shap].values, X_raw.iloc[val_idx_shap].values,
        df_aa, df_miss, df_cat, df_tda, clean_shap, val_idx_shap)
    shap_sample_idx = np.random.choice(len(clean_shap), min(500,len(clean_shap)), replace=False)
    X_shap_df = pd.DataFrame(Xtr_shap[shap_sample_idx],
                             columns=feat_names[:Xtr_shap.shape[1]])
    plot_shap(models_xgb[best_fold_idx], X_shap_df,
              save_path='outputs/shap.png')

    print("\n=== 9. KLİNİK RAPORLAR ===")
    explainer  = shap.TreeExplainer(models_xgb[best_fold_idx])
    shap_vals  = explainer.shap_values(X_shap_df)
    feat_names_list = list(X_shap_df.columns)

    for pname in ['master', 'kanser', 'pah', 'cftr']:
        idx = panels[panels == pname].index[0]
        if idx in shap_sample_idx.tolist():
            panel_shap_idx = shap_sample_idx.tolist().index(idx)
        else:
            panel_candidates = [
                i for i, si in enumerate(shap_sample_idx)
                if panels.iloc[si] == pname
            ]
            panel_shap_idx = panel_candidates[0] if panel_candidates else 0

        klinik_rapor_uret(
            variant_id    = f"{pname.upper()}_ORNEK",
            panel         = pname,
            proba         = float(oof_stack[idx]),
            shap_values   = shap_vals[panel_shap_idx],
            feature_names = feat_names_list,
            aa1           = str(df['AA_1'].iloc[idx]),
            aa2           = str(df['AA_2'].iloc[idx]),
            grantham      = float(GRANTHAM.get(
                                (str(df['AA_1'].iloc[idx]),
                                 str(df['AA_2'].iloc[idx])), 0))
        )

    print("4 panel raporu olusturuldu.")

    # ════════════════════════════════════════════════════
    # TEST INFERENCE — yarışma günü bu satırı aktif et
    # ════════════════════════════════════════════════════
    # predict_test(
    #     test_csv_path = 'YARISMA_TEST.csv',
    #     models_xgb    = models_xgb,
    #     models_lgb    = models_lgb,
    #     models_ft     = models_ft,
    #     meta_learner  = meta_learner,
    #     thr_stack     = thr_stack,
    #     ohe_dict      = ohe_dict,
    #     feat_names    = feat_names,
    #     X_raw_train   = X_raw,
    #     df_miss_cols  = df_miss.columns,
    #     df_cat_cols   = df_cat.columns,
    # )

    print("\n=== ÖZET ===")
    print(f"  Stacking OOF PR-AUC : {m_stack['pr_auc']:.4f}")
    print(f"  Stacking OOF ROC-AUC: {m_stack['roc_auc']:.4f}  "
          f"[%95 GA: {ci['roc_auc'][1]:.4f}–{ci['roc_auc'][2]:.4f}]")
    print(f"  Stacking OOF F2     : {m_stack['f2']:.4f}  "
          f"[%95 GA: {ci['f2'][1]:.4f}–{ci['f2'][2]:.4f}]")
    print(f"  Stacking OOF MCC    : {m_stack['mcc']:.4f}  "
          f"[%95 GA: {ci['mcc'][1]:.4f}–{ci['mcc'][2]:.4f}]")
    print(f"  Stacking OOF Recall : {m_stack['recall']:.4f}")
    print(f"  Optimal Eşik        : {thr_stack:.3f}")
    print(f"  TDA Özellik Sayısı  : {len(TDA_NAMES)}")
    print(f"  Yapısal Özellikler  : {len(STRUCT_FEAT_NAMES)}")
    print(f"  Toplam Özellik      : {len(feat_names)}")
    print(f"  Çıktılar            : outputs/")

In [ ]:
# ════════════════════════════════════════════════════════
# GRADIO İNTERAKTİF DEMO
# Bu hücreyi __main__ bloğu ÇALIŞTIKTAN SONRA (model eğitildikten sonra) çalıştırın.
# Eğitilmiş modelleri (models_xgb, models_lgb, models_ft, meta_learner,
# thr_stack, ohe_dict) bellekteki haliyle kullanır.
# ════════════════════════════════════════════════════════


import gradio as gr
import numpy as np
import pandas as pd
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler

print("[DEMO] Inference için ön işleme bileşenleri hazırlanıyor...")

# ── Tüm eğitim verisi üzerinden global imputer + scaler fit et ──
# (Demo amaçlı; yarışma günü predict_test() fonksiyonu kullanılır)
_demo_imputer = IterativeImputer(random_state=RANDOM_SEED, max_iter=7,
                                  n_nearest_features=20, sample_posterior=True)
_demo_imputer.fit(X_raw.values)
_demo_sc_num = RobustScaler().fit(_demo_imputer.transform(X_raw.values))
_demo_sc_aa  = RobustScaler().fit(df_aa.values)
_demo_sc_tda = RobustScaler().fit(df_tda.values)

# SHAP açıklayıcı (en iyi fold'un XGB modeli)
_demo_best_fold = int(np.argmax([f['metrics']['pr_auc'] for f in fold_results]))
_demo_explainer = shap.TreeExplainer(models_xgb[_demo_best_fold])
_demo_feat_names = (list(X_raw.columns) + AA_FEAT_NAMES + STRUCT_FEAT_NAMES +
                    TDA_NAMES + list(df_miss.columns) + list(df_cat.columns))

_AA_FULL = {
    'A':'Alanin','C':'Sistein','D':'Aspartik asit','E':'Glutamik asit',
    'F':'Fenilalanin','G':'Glisin','H':'Histidin','I':'İzolösin','K':'Lizin',
    'L':'Lösin','M':'Metiyonin','N':'Asparajin','P':'Prolin','Q':'Glutamin',
    'R':'Arjinin','S':'Serin','T':'Treonin','V':'Valin','W':'Triptofan','Y':'Tirozin'
}
_AA_CHOICES = [f"{k} — {v}" for k, v in _AA_FULL.items()]


def _build_demo_vector(aa1, aa2, panel_name):
    """Tek bir varyant için tam özellik vektörü kurar (eğitimle aynı sıra)."""
    n_num = X_raw.shape[1]

    # Sayısal kısım: medyan ile doldur (tek satır impute edilemez), sonra ölçekle
    x_num = X_raw.median().values.reshape(1, -1)
    x_num_sc = _demo_sc_num.transform(x_num)

    # AA biyokimyasal + yapısal
    aa_vec = np.concatenate([aa_features(aa1, aa2),
                             structural_features(aa1, aa2)]).reshape(1, -1)
    aa_sc = _demo_sc_aa.transform(aa_vec)

    # TDA — tek satırdan anlamlı TDA çıkmaz, sıfır vektör (eğitimdeki ölçekle)
    tda_vec = np.zeros((1, len(TDA_NAMES)), dtype=np.float32)
    tda_sc = _demo_sc_tda.transform(tda_vec)

    # Eksiklik göstergeleri ve kategorik: sıfır (demo varsayımı)
    miss_vec = np.zeros((1, df_miss.shape[1]), dtype=np.float32)
    cat_vec  = np.zeros((1, df_cat.shape[1]), dtype=np.float32)

    X_full = np.hstack([x_num_sc, aa_sc, tda_sc, miss_vec, cat_vec]).astype(np.float32)
    return X_full


def predict_variant_demo(aa1_full, aa2_full, panel_label, make_pdf):
    aa1 = aa1_full.split(' — ')[0].strip().upper()
    aa2 = aa2_full.split(' — ')[0].strip().upper()
    panel = panel_label.lower()

    if aa1 == aa2:
        return ("⚠ Referans ve mutant amino asit aynı — bu bir varyant değil.",
                None, None)

    X_full = _build_demo_vector(aa1, aa2, panel)

    # Ensemble tahmin (stacking)
    p_xgb = np.mean([m.predict_proba(X_full)[:, 1] for m in models_xgb])
    p_lgb = np.mean([m.predict_proba(X_full)[:, 1] for m in models_lgb])
    p_ft  = np.mean([m.predict_proba(X_full) for m in models_ft])
    meta_in = np.array([[p_ft, p_xgb, p_lgb]])
    proba = float(meta_learner.predict_proba(meta_in)[:, 1][0])

    karar = "PATOJENİK" if proba >= thr_stack else "BENIGN"
    guven = "YÜKSEK" if abs(proba - 0.5) > 0.25 else "ORTA"
    gran = float(GRANTHAM.get((aa1, aa2), 0))

    # SHAP — en etkili 5 faktör
    shap_vals = _demo_explainer.shap_values(X_full)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1] if len(shap_vals) > 1 else shap_vals[0]
    shap_row = np.array(shap_vals).flatten()
    top5_idx = np.argsort(np.abs(shap_row))[-5:][::-1]

    # Sonuç metni
    renk = "🔴" if karar == "PATOJENİK" else "🟢"
    md = f"""
## {renk} Karar: **{karar}**

| | |
|---|---|
| **Patojenik Olasılığı** | {proba:.1%} |
| **Güven Seviyesi** | {guven} |
| **Karar Eşiği** | {thr_stack:.3f} |
| **AA Değişimi** | {aa1} ({_AA_FULL[aa1]}) → {aa2} ({_AA_FULL[aa2]}) |
| **Grantham Mesafesi** | {gran:.0f} {'(radikal)' if gran>100 else '(orta)' if gran>50 else '(benzer)'} |
| **Panel** | {panel.upper()} — {PANEL_PHENOTYPE.get(panel,'')} |

**Klinik Öneri:** {PANEL_ACTION.get(panel, 'Klinisyen değerlendirmesi önerilir.')}

---
### Kararı Etkileyen Başlıca Faktörler (SHAP)
"""
    for idx in top5_idx:
        if idx < len(_demo_feat_names):
            fname = _demo_feat_names[idx]
            yon = "patojenik ↑" if shap_row[idx] > 0 else "benign ↓"
            md += f"- **{fname}**: {yon} (etki {shap_row[idx]:+.3f})\n"

    md += f"\n*Not: Tarama amaçlı karar-destek çıktısıdır. Nihai tanı klinisyene aittir.*"

    # Olasılık göstergesi (basit bar)
    bar_data = {"Benign": 1 - proba, "Patojenik": proba}

    pdf_path = None
    if make_pdf:
        try:
            klinik_rapor_uret(
                variant_id=f"DEMO_{aa1}{aa2}_{panel.upper()}",
                panel=panel, proba=proba,
                shap_values=shap_row, feature_names=_demo_feat_names,
                aa1=aa1, aa2=aa2, grantham=gran)
            pdf_path = f"rapor_DEMO_{aa1}{aa2}_{panel.upper()}.pdf"
        except Exception as e:
            md += f"\n\n⚠ PDF üretilemedi: {e}"

    return md, bar_data, pdf_path


# ── Arayüz ──
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue"),
               title="VarPath Varyant Sınıflandırıcı") as demo:
    gr.Markdown(
        "# 🧬 VarPath — Genetik Varyant Patojenisite Sınıflandırıcı\n"
        "Bir missense varyantın (amino asit değişiminin) patojenik mi benign mi "
        "olduğunu tahmin eder. Stacking ensemble (XGBoost + LightGBM + FT-Transformer)."
    )
    with gr.Row():
        with gr.Column(scale=1):
            aa1_in = gr.Dropdown(_AA_CHOICES, value=_AA_CHOICES[16],
                                 label="Referans Amino Asit (orijinal)")
            aa2_in = gr.Dropdown(_AA_CHOICES, value=_AA_CHOICES[18],
                                 label="Mutant Amino Asit (değişen)")
            panel_in = gr.Radio(["MASTER", "KANSER", "PAH", "CFTR"],
                                value="KANSER", label="Panel")
            pdf_in = gr.Checkbox(label="PDF klinik rapor üret", value=False)
            btn = gr.Button("🔬 Analiz Et", variant="primary", size="lg")
            gr.Markdown(
                "**Örnek deneyin:** R→W (KANSER) patojenik eğilimli · "
                "A→S benign eğilimli"
            )
        with gr.Column(scale=2):
            out_md = gr.Markdown()
            out_bar = gr.Label(label="Olasılık Dağılımı")
            out_pdf = gr.File(label="Klinik Rapor (PDF)")

    btn.click(predict_variant_demo,
              inputs=[aa1_in, aa2_in, panel_in, pdf_in],
              outputs=[out_md, out_bar, out_pdf])

print("[DEMO] Başlatılıyor — aşağıdaki public linke tıklayın")
demo.launch(share=True)
